In [0]:
spark

In [0]:
# 1. Read the dataset from your storage volume path
df = spark.read.csv("/Volumes/students/default/filestore/Online_Retail-1.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5, truncate=False)
print(f"Total transactions: {df.count()}")

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price_krw: integer (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- sales_amount_krw: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- returned: string (nullable = true)
 |-- rating: integer (nullable = true)

+--------+----------+-----------+-------+----------------+----------+------------+-----------+--------+--------------+-------------+----------------+----------+--------------+--------+------+
|order_id|order_date|customer_id|city   |customer_segment|product_id|product_name|category   |quantity|unit_price_krw|discou

In [0]:
# 2. Data cleaning and preprocessing transformations
from pyspark.sql.functions import col, to_timestamp, when, lit, regexp_replace

df_clean = df.filter(col("CustomerID").isNotNull())

# 1. First, reorganize the raw string format to match your professor's string requirements
# This safely converts '13/12/2010 9:09' to '12/13/2010 9:09'
df_clean = df_clean.withColumn(
    "InvoiceDate", 
    regexp_replace(col("InvoiceDate"), r"^(\d+)/(\d+)/", r"$2/$1/")
)

# 2. Now your professor's exact format string works perfectly without crashing!
df_clean = df_clean.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm"))
df_clean = df_clean.withColumn("TotalPrice", col("Quantity") * col("UnitPrice"))

df_clean.show(5, truncate=False)
df_clean.printSchema()

df_clean = df_clean.withColumn("IsUK", when(col("Country") == "United Kingdom", lit(1)).otherwise(lit(0)))

df_clean.show(5, truncate=False)
display(df_clean)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5813757425934223>, line 17
     14 df_clean = df_clean.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm"))
     15 df_clean = df_clean.withColumn("TotalPrice", col("Quantity") * col("UnitPrice"))
---> 17 df_clean.show(5, truncate=False)
     18 df_clean.printSchema()
     20 df_clean = df_clean.withColumn("IsUK", when(col("Country") == "United Kingdom", lit(1)).otherwise(lit(0)))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self

In [0]:
# 3. Filtering for high value transactions
high_value_df = df_clean.filter((col("TotalPrice") > 50) & (col("Quantity") > 10))
high_value_df.display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5813757425934224>, line 3
      1 # 3. Filtering for high value transactions
      2 high_value_df = df_clean.filter((col("TotalPrice") > 50) & (col("Quantity") > 10))
----> 3 high_value_df.display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:80, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     76 def df_display(df, *args, **kwargs):
     77     """
     78     df.display() is an alias for display(df). Run help(display) for more information.
     79     """
---> 80     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isi

In [0]:
# 4. Grouping by items to get transaction metrics
product_stats_df = df_clean.groupBy("StockCode", "Description").agg(
    {"Quantity": "sum", "TotalPrice": "avg", "InvoiceNo": "count"}
).withColumnRenamed("sum(Quantity)", "total_quantity") \
 .withColumnRenamed("avg(TotalPrice)", "avg_price") \
 .withColumnRenamed("count(InvoiceNo)", "transaction_count")

product_stats_df.orderBy(col("total_quantity").desc()).display()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# 5. Extract distinct customer mappings
customers_df = df_clean.select("CustomerID", "Country").distinct()
customers_df.display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5813757425934236>, line 3
      1 # 5. Extract distinct customer mappings
      2 customers_df = df_clean.select("CustomerID", "Country").distinct()
----> 3 customers_df.display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:80, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     76 def df_display(df, *args, **kwargs):
     77     """
     78     df.display() is an alias for display(df). Run help(display) for more information.
     79     """
---> 80     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, Conne

In [0]:
# 6. Join the customer dataset
customer_transactions_df = df_clean.join(
    customers_df.select([i for i in customers_df.columns if i != 'Country']), 
    "CustomerID", 
    "inner"
)
customer_transactions_df.display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5813757425934237>, line 3
      1 # 6. Join the customer dataset
      2 customer_transactions_df = df_clean.join(
----> 3     customers_df.select([i for i in customers_df.columns if i != 'Country']), 
      4     "CustomerID", 
      5     "inner"
      6 )
      7 customer_transactions_df.display()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:309, in DataFrame.columns(self)
    307 @property
    308 def columns(self) -> List[str]:
--> 309     return [field.name for field in self._schema.fields]

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1977, in DataFrame._schema(self)
   1975 if self._cached_schema is None:
   1976     query = self._plan.to_proto(self._session.client)
-> 1977     self._cached_schema = self._session.client.schema(query)

In [0]:
# 7. Aggregate spending patterns per customer
# Note: Legacy .cache() call removed here to comply with serverless execution
customer_summary_df = customer_transactions_df.groupBy("CustomerID", "Country").agg(
    {"TotalPrice": "sum", "InvoiceNo": "count"}
).withColumnRenamed("sum(TotalPrice)", "total_spent") \
 .withColumnRenamed("count(InvoiceNo)", "purchase_count")

customer_summary_df.orderBy(col("total_spent").desc()).display()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5813757425934238>, line 3
      1 # 7. Aggregate spending patterns per customer
      2 # Note: Legacy .cache() call removed here to comply with serverless execution
----> 3 customer_summary_df = customer_transactions_df.groupBy("CustomerID", "Country").agg(
      4     {"TotalPrice": "sum", "InvoiceNo": "count"}
      5 ).withColumnRenamed("sum(TotalPrice)", "total_spent") \
      6  .withColumnRenamed("count(InvoiceNo)", "purchase_count")
      8 customer_summary_df.orderBy(col("total_spent").desc()).display()

NameError: name 'customer_transactions_df' is not defined

In [0]:
# 8. Create temporary view for Spark SQL execution
df_clean.createOrReplaceTempView("retail")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5813757425934239>, line 2
      1 # 8. Create temporary view for Spark SQL execution
----> 2 df_clean.createOrReplaceTempView("retail")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2115, in DataFrame.createOrReplaceTempView(self, name)
   2111 def createOrReplaceTempView(self, name: str) -> None:
   2112     command = plan.CreateView(
   2113         child=self._plan, name=name, is_global=False, replace=True
   2114     ).command(session=self._session.client)
-> 2115     _, _, ei = self._session.client.execute_command(command, self._plan.observations)
   2116     self._execution_info = ei

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536   

In [0]:
# 9. Simple bulk order filter query
basic_sql = spark.sql("""
    select CustomerID, Description, Quantity, UnitPrice
    from retail
    where Quantity > 80000
    order by Quantity desc
""")
basic_sql.display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5813757425934240>, line 2
      1 # 9. Simple bulk order filter query
----> 2 basic_sql = spark.sql("""
      3     select CustomerID, Description, Quantity, UnitPrice
      4     from retail
      5     where Quantity > 80000
      6     order by Quantity desc
      7 """)
      8 basic_sql.display()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_command_result" in properties:
    903     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/py

In [0]:
# 10. Map customer temporary view
df_clean.createOrReplaceTempView("customers")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# 11. Country-level metric analysis query
# Fixed: Rectified alias spacing typo and matched column casing
country_summary_sql = spark.sql("""
    select c.Country, COUNT(distinct r.InvoiceNo) as invoice_count, round(sum(r.Quantity * r.UnitPrice), 2) as total_revenue
    from retail r
    join customers c ON r.CustomerID = c.CustomerID
    group by c.Country
    order by total_revenue desc
""")
country_summary_sql.display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5813757425934242>, line 3
      1 # 11. Country-level metric analysis query
      2 # Fixed: Rectified alias spacing typo and matched column casing
----> 3 country_summary_sql = spark.sql("""
      4     select c.Country, COUNT(distinct r.InvoiceNo) as invoice_count, round(sum(r.Quantity * r.UnitPrice), 2) as total_revenue
      5     from retail r
      6     join customers c ON r.CustomerID = c.CustomerID
      7     group by c.Country
      8     order by total_revenue desc
      9 """)
     10 country_summary_sql.display()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client

In [0]:
# 12. Segment identification query
customes_pattern_sql = spark.sql("""
    select CustomerID, COUNT(distinct StockCode) as unique_products, AVG(Quantity) as avg_quantity
    from retail
    group by CustomerID
    having count(distinct StockCode) > 10
    order by unique_products desc
""")
customes_pattern_sql.display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5813757425934243>, line 2
      1 # 12. Segment identification query
----> 2 customes_pattern_sql = spark.sql("""
      3     select CustomerID, COUNT(distinct StockCode) as unique_products, AVG(Quantity) as avg_quantity
      4     from retail
      5     group by CustomerID
      6     having count(distinct StockCode) > 10
      7     order by unique_products desc
      8 """)
      9 customes_pattern_sql.display()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_command_result" in properties:
    903     df = Data

In [0]:
# 13. Temporal revenue tracking query
# Fixed: Rectified misspelled group by column name and alias spacing typo
time_analysis_sql = spark.sql("""
    select YEAR(InvoiceDate) as year, MONTH(InvoiceDate) as month, SUM(TotalPrice) as monthly_revenue
    from retail
    group by YEAR(InvoiceDate), MONTH(InvoiceDate)
    order by year, month
""")
time_analysis_sql.display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5813757425934244>, line 3
      1 # 13. Temporal revenue tracking query
      2 # Fixed: Rectified misspelled group by column name and alias spacing typo
----> 3 time_analysis_sql = spark.sql("""
      4     select YEAR(InvoiceDate) as year, MONTH(InvoiceDate) as month, SUM(TotalPrice) as monthly_revenue
      5     from retail
      6     group by YEAR(InvoiceDate), MONTH(InvoiceDate)
      7     order by year, month
      8 """)
      9 time_analysis_sql.display()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_co